[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_09/notebook_9_3_bias_mitigation.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 9.3: Bias Mitigation Strategies

**Chapter 9: Fairness and Bias in Healthcare AI**

In this notebook, we implement and compare three major approaches to bias mitigation:

1. **Pre-processing**: Modify training data before model training
2. **In-processing**: Incorporate fairness constraints during training
3. **Post-processing**: Adjust model predictions after training

We'll apply these techniques to a realistic healthcare scenario: melanoma detection across different skin tones.

## Learning Objectives

By the end of this notebook, you will:
- Understand the three main categories of bias mitigation
- Implement resampling and SMOTE for data balancing
- Train fairness-constrained models with custom loss functions
- Apply threshold optimization for post-processing fairness
- Compare mitigation strategies on clinical metrics
- Understand fairness-accuracy trade-offs

## Clinical Context

**Problem**: Melanoma detection models trained on predominantly light-skinned patients achieve:
- Sensitivity: 94.2% on light skin, 68.5% on dark skin (the figures reported in Chapter 9 for Priya's app)
- This 26 percentage point gap means the model missed nearly one-third of melanomas in patients with darker skin

*Our synthetic dataset below produces its own (smaller or larger) gap; all numbers in this notebook are computed from it.*

**Goal**: Apply bias mitigation to improve sensitivity on dark skin while maintaining overall clinical utility.

---

## Part 1: Setup and Baseline Model

First, let's create a realistic biased dataset and establish baseline performance.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
from sklearn.utils import resample
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

print("✓ Libraries imported successfully")
print("\nPackages used:")
print("- NumPy, Pandas: Data manipulation")
print("- Scikit-learn: Machine learning models")
print("- imbalanced-learn: SMOTE implementation")
print("- Matplotlib, Seaborn: Visualization")

✓ Libraries imported successfully

Packages used:
- NumPy, Pandas: Data manipulation
- Scikit-learn: Machine learning models
- imbalanced-learn: SMOTE implementation
- Matplotlib, Seaborn: Visualization


### Create Realistic Biased Dataset

We'll create a melanoma detection dataset with:
- **Representation bias**: 92% light skin (Fitzpatrick I–III), 8% dark skin (IV–VI), as in the dataset behind Priya's app (Chapter 9)
- **Feature/measurement bias**: lower lesion–skin contrast on dark skin weakens the visual melanoma cues (by 25%) and makes them noisier to measure
- **Label quality bias**: historical labels miss more melanomas on dark skin (5% vs 2% labelled benign), reflecting underdiagnosis

These are generator *assumptions* (knobs at the top of the function), chosen so the baseline lands near the book's 94% vs 68.5% sensitivity on average; they are not measured values.

In [2]:
def create_melanoma_dataset(n_samples=5000):
    """
    Create synthetic melanoma detection dataset with realistic bias.

    Features (simulating dermatoscopy measurements):
    - asymmetry: 0-1 (lesion asymmetry)
    - border_irregularity: 0-1
    - color_variation: 0-1
    - diameter: mm
    - redness: 0-1 (harder to measure on dark skin)
    - pigment_network: 0-1

    Protected attribute:
    - skin_tone: 'light' or 'dark'

    Label:
    - malignant: 0 (benign) or 1 (melanoma)
    """
    # Representation bias: 92% light (I-III), 8% dark (IV-VI)
    n_light = int(n_samples * 0.92)
    n_dark = n_samples - n_light

    skin_tone = ['light'] * n_light + ['dark'] * n_dark

    # True melanoma prevalence (similar across groups)
    melanoma_rate = 0.15

    # Bias MECHANISM knobs (assumptions of this synthetic generator, not measured values)
    SEPARATION = 0.6        # melanomas are often subtle: features only 60% of the way from benign to classic
    DARK_CONTRAST = 0.75    # lower lesion-skin contrast: visual melanoma cues 25% weaker on dark skin
    MEASURE_SD = 0.10       # measurement noise on visual features, everyone
    DARK_EXTRA_SD = 0.16    # extra measurement noise on dark skin (harder to image and grade)
    MISSED_LABEL = {'light': 0.02, 'dark': 0.05}   # melanomas historically labelled benign (underdiagnosis)
    FALSE_MELANOMA_LABEL = 0.005                    # benign labelled melanoma (biopsy-confirmed labels are rarely wrong)
    BENIGN_MEAN = {'asymmetry': 0.2, 'border_irregularity': 2/9, 'color_variation': 0.3,
                   'redness': 1/3, 'pigment_network': 0.7}

    data = []

    for i, tone in enumerate(skin_tone):
        # Ground truth: is this actually melanoma?
        is_melanoma = np.random.random() < melanoma_rate

        if is_melanoma:
            # Melanoma features (more extreme values)
            asymmetry = np.random.beta(8, 2)  # Skewed toward 1
            border_irregularity = np.random.beta(7, 2)
            color_variation = np.random.beta(7, 3)
            diameter = np.random.normal(8, 2)  # Larger
            redness = np.random.beta(6, 3)
            pigment_network = np.random.beta(3, 7)  # Atypical
        else:
            # Benign features (more normal values)
            asymmetry = np.random.beta(2, 8)  # Skewed toward 0
            border_irregularity = np.random.beta(2, 7)
            color_variation = np.random.beta(3, 7)
            diameter = np.random.normal(4, 1.5)  # Smaller
            redness = np.random.beta(3, 6)
            pigment_network = np.random.beta(7, 3)  # Typical

        visual = {'asymmetry': asymmetry, 'border_irregularity': border_irregularity,
                  'color_variation': color_variation, 'redness': redness, 'pigment_network': pigment_network}
        if is_melanoma:
            diameter = 4 + SEPARATION * (diameter - 4)
            for k in visual:
                # Subtle melanomas; on dark skin the visual contrast is further reduced
                shrink = SEPARATION * (DARK_CONTRAST if tone == 'dark' else 1.0)
                visual[k] = BENIGN_MEAN[k] + shrink * (visual[k] - BENIGN_MEAN[k])

        # Measurement bias: visual features are noisier to measure, more so on dark skin
        sd = np.sqrt(MEASURE_SD**2 + (DARK_EXTRA_SD**2 if tone == 'dark' else 0))
        for k in visual:
            visual[k] = np.clip(visual[k] + np.random.normal(0, sd), 0, 1)

        # Label quality bias: historical labels MISS more melanomas on dark skin (asymmetric noise)
        label = is_melanoma
        if is_melanoma and np.random.random() < MISSED_LABEL[tone]:
            label = False
        elif not is_melanoma and np.random.random() < FALSE_MELANOMA_LABEL:
            label = True

        data.append({
            **visual,
            'diameter': diameter,
            'skin_tone': tone,
            'malignant': int(label)
        })

    df = pd.DataFrame(data)[['asymmetry', 'border_irregularity', 'color_variation', 'diameter',
                             'redness', 'pigment_network', 'skin_tone', 'malignant']]

    # Clip all features to valid ranges
    for col in ['asymmetry', 'border_irregularity', 'color_variation', 'redness', 'pigment_network']:
        df[col] = df[col].clip(0, 1)
    df['diameter'] = df['diameter'].clip(1, 20)

    return df

# Create dataset
df = create_melanoma_dataset(n_samples=20000)  # 20k so the 8% dark-skin group has enough melanomas to tune and evaluate

print("Dataset created successfully!")
print(f"\nTotal samples: {len(df)}")
print(f"\nSkin tone distribution:")
print(df['skin_tone'].value_counts())
print(f"\nMelanoma prevalence:")
print(df.groupby('skin_tone')['malignant'].agg(['sum', 'mean']))
print("\nFirst few rows:")
df.head()

Dataset created successfully!

Total samples: 20000

Skin tone distribution:
skin_tone
light    18400
dark      1600
Name: count, dtype: int64

Melanoma prevalence:
            sum      mean
skin_tone                
dark        204  0.127500
light      2802  0.152283

First few rows:


,asymmetry,border_irregularity,color_variation,diameter,redness,pigment_network,skin_tone,malignant
0,0.043149,0.431694,0.292116,1.000000,0.571191,0.510733,light,0
1,0.443355,0.615288,0.453120,9.733976,0.412167,0.382086,light,1
2,0.685095,0.943285,0.277323,7.815660,0.451953,0.501279,light,1
3,0.384406,0.776119,0.541687,7.001738,0.362590,0.565153,light,1
4,0.671303,0.319032,0.796525,6.522817,0.672963,0.563532,light,1


### Visualize Dataset Characteristics

In [3]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

features = ['asymmetry', 'border_irregularity', 'color_variation',
            'diameter', 'redness', 'pigment_network']

for idx, feature in enumerate(features):
    ax = axes[idx // 3, idx % 3]

    # Plot distributions by skin tone and malignancy
    for tone in ['light', 'dark']:
        for malignant in [0, 1]:
            subset = df[(df['skin_tone'] == tone) & (df['malignant'] == malignant)]
            label = f"{tone.capitalize()} - {'Melanoma' if malignant else 'Benign'}"
            alpha = 0.5 if malignant == 0 else 0.7
            linestyle = '-' if tone == 'light' else '--'
            ax.hist(subset[feature], bins=30, alpha=alpha,
                   label=label, linestyle=linestyle, histtype='step', linewidth=2)

    ax.set_xlabel(feature.replace('_', ' ').title())
    ax.set_ylabel('Frequency')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('dataset_feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey observations:")
print("1. Malignant lesions have higher asymmetry, border irregularity, and color variation")
print("2. On dark skin, melanoma colour/shape cues sit closer to benign values and are noisier (dashed lines)")
print("3. These biases will affect model performance differently across groups")


Key observations:
1. Malignant lesions have higher asymmetry, border irregularity, and color variation
2. On dark skin, melanoma colour/shape cues sit closer to benign values and are noisier (dashed lines)
3. These biases will affect model performance differently across groups


### Train Baseline Model (No Mitigation)

Let's establish baseline performance to understand the bias problem.

In [4]:
# Prepare features and labels
feature_cols = ['asymmetry', 'border_irregularity', 'color_variation',
                'diameter', 'redness', 'pigment_network']

X = df[feature_cols].values
y = df['malignant'].values
groups = df['skin_tone'].values

# Split into train/test (stratified by both label and group)
X_train, X_test, y_train, y_test, groups_train, groups_test = train_test_split(
    X, y, groups, test_size=0.3, random_state=42, stratify=y
)

# Hold out a VALIDATION set from the training data: post-processing thresholds are tuned
# on it, so the test set stays untouched until final evaluation
X_train, X_val, y_train, y_val, groups_train, groups_val = train_test_split(
    X_train, y_train, groups_train, test_size=0.25, random_state=42, stratify=y_train
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)} (dark skin: {(groups_val == 'dark').sum()})")
print(f"Test samples: {len(X_test)}")
print(f"\nTraining set skin tone distribution:")
print(pd.Series(groups_train).value_counts())
print(f"\nTest set skin tone distribution:")
print(pd.Series(groups_test).value_counts())

Training samples: 10500
Validation samples: 3500 (dark skin: 267)
Test samples: 6000

Training set skin tone distribution:
light    9648
dark      852
Name: count, dtype: int64

Test set skin tone distribution:
light    5519
dark      481
Name: count, dtype: int64


In [5]:
# Train baseline model
baseline_model = LogisticRegression(random_state=42, max_iter=1000)
baseline_model.fit(X_train, y_train)

# Predictions
y_pred_baseline = baseline_model.predict(X_test)
y_prob_baseline = baseline_model.predict_proba(X_test)[:, 1]

print("✓ Baseline model trained")
print(f"\nOverall Test Accuracy: {(y_pred_baseline == y_test).mean():.3f}")
print(f"Overall Test AUC: {roc_auc_score(y_test, y_prob_baseline):.3f}")

✓ Baseline model trained

Overall Test Accuracy: 0.981
Overall Test AUC: 0.983


### Evaluate Baseline by Subgroup

The key question: does performance differ between light and dark skin?

In [6]:
def evaluate_fairness(y_true, y_pred, y_prob, groups, model_name="Model"):
    """
    Comprehensive fairness evaluation across subgroups.
    """
    results = {}

    for group in np.unique(groups):
        mask = groups == group
        y_true_group = y_true[mask]
        y_pred_group = y_pred[mask]
        y_prob_group = y_prob[mask]

        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(y_true_group, y_pred_group).ravel()

        # Clinical metrics
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0  # Recall
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0  # Precision
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0
        accuracy = (tp + tn) / (tp + tn + fp + fn)

        # AUC
        auc = roc_auc_score(y_true_group, y_prob_group) if len(np.unique(y_true_group)) > 1 else np.nan

        # Selection rate (positive prediction rate)
        selection_rate = y_pred_group.mean()

        results[group] = {
            'n': len(y_true_group),
            'prevalence': y_true_group.mean(),
            'sensitivity': sensitivity,
            'specificity': specificity,
            'ppv': ppv,
            'npv': npv,
            'accuracy': accuracy,
            'auc': auc,
            'selection_rate': selection_rate,
            'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn
        }

    # Print results
    print(f"\n{'='*70}")
    print(f"{model_name} - Fairness Evaluation")
    print(f"{'='*70}\n")

    for group in results:
        r = results[group]
        print(f"{group.upper()} SKIN (n={r['n']})")
        print(f"  Prevalence: {r['prevalence']:.3f}")
        print(f"  Sensitivity (TPR): {r['sensitivity']:.3f}  ⭐ CRITICAL for cancer detection")
        print(f"  Specificity (TNR): {r['specificity']:.3f}")
        print(f"  PPV (Precision): {r['ppv']:.3f}")
        print(f"  NPV: {r['npv']:.3f}")
        print(f"  Accuracy: {r['accuracy']:.3f}")
        print(f"  AUC: {r['auc']:.3f}")
        print(f"  Selection Rate: {r['selection_rate']:.3f}")
        print(f"  Confusion: TP={r['tp']}, FP={r['fp']}, FN={r['fn']}, TN={r['tn']}")
        print()

    # Disparities
    groups_list = list(results.keys())
    if len(groups_list) == 2:
        g1, g2 = groups_list
        sens_gap = abs(results[g1]['sensitivity'] - results[g2]['sensitivity'])
        spec_gap = abs(results[g1]['specificity'] - results[g2]['specificity'])
        ppv_gap = abs(results[g1]['ppv'] - results[g2]['ppv'])
        auc_gap = abs(results[g1]['auc'] - results[g2]['auc'])

        print(f"FAIRNESS GAPS:")
        print(f"  Sensitivity gap: {sens_gap:.3f} {'✓ OK' if sens_gap < 0.05 else '⚠️ LARGE'}")
        print(f"  Specificity gap: {spec_gap:.3f} {'✓ OK' if spec_gap < 0.05 else '⚠️ LARGE'}")
        print(f"  PPV gap: {ppv_gap:.3f} {'✓ OK' if ppv_gap < 0.05 else '⚠️ LARGE'}")
        print(f"  AUC gap: {auc_gap:.3f} {'✓ OK' if auc_gap < 0.03 else '⚠️ LARGE'}")
        print()

        if sens_gap > 0.05:
            print("⚠️ CLINICAL CONCERN: Large sensitivity gap detected!")
            print("   This means melanoma detection rates differ significantly by skin tone.")
            print("   Bias mitigation is strongly recommended.\n")

    # Small gaps can come from a useless model: check every group still gets a real decision
    for group, r in results.items():
        if r['selection_rate'] <= 0.01 or r['selection_rate'] >= 0.99:
            print(f"⚠️ DEGENERATE for {group} skin: the model flags {r['selection_rate']:.0%} of patients "
                  f"(sensitivity {r['sensitivity']:.2f}, specificity {r['specificity']:.2f}).")
            print("   Equal metrics across groups mean nothing clinically if the model flags everyone or no one.\n")

    return results

# Evaluate baseline
baseline_results = evaluate_fairness(
    y_test, y_pred_baseline, y_prob_baseline, groups_test,
    model_name="BASELINE MODEL (No Mitigation)"
)


BASELINE MODEL (No Mitigation) - Fairness Evaluation

DARK SKIN (n=481)
  Prevalence: 0.100
  Sensitivity (TPR): 0.708  ⭐ CRITICAL for cancer detection
  Specificity (TNR): 0.975
  PPV (Precision): 0.756
  NPV: 0.968
  Accuracy: 0.948
  AUC: 0.956
  Selection Rate: 0.094
  Confusion: TP=34, FP=11, FN=14, TN=422

LIGHT SKIN (n=5519)
  Prevalence: 0.155
  Sensitivity (TPR): 0.943  ⭐ CRITICAL for cancer detection
  Specificity (TNR): 0.991
  PPV (Precision): 0.952
  NPV: 0.990
  Accuracy: 0.984
  AUC: 0.984
  Selection Rate: 0.153
  Confusion: TP=805, FP=41, FN=49, TN=4624

FAIRNESS GAPS:
  Sensitivity gap: 0.234 ⚠️ LARGE
  Specificity gap: 0.017 ✓ OK
  PPV gap: 0.196 ⚠️ LARGE
  AUC gap: 0.028 ✓ OK

⚠️ CLINICAL CONCERN: Large sensitivity gap detected!
   This means melanoma detection rates differ significantly by skin tone.
   Bias mitigation is strongly recommended.



## Part 2: Pre-Processing Mitigation

**Strategy**: Modify the training data before model training to reduce bias.

### Technique 1: Simple Resampling

Balance representation by:
- Oversampling minority group (dark skin)
- Undersampling majority group (light skin)
- Or both

In [7]:
def resample_for_fairness(X, y, groups, strategy='oversample'):
    """
    Resample training data to balance group representation.

    Parameters:
    - strategy: 'oversample' (increase minority), 'undersample' (decrease majority),
                or 'balanced' (both)
    """
    # Combine into DataFrame for easier manipulation
    df_train = pd.DataFrame(X, columns=feature_cols)
    df_train['label'] = y
    df_train['group'] = groups

    # Current group sizes
    group_counts = df_train['group'].value_counts()
    print(f"Original training set:")
    print(group_counts)

    # Determine target size
    if strategy == 'oversample':
        target_size = group_counts.max()
    elif strategy == 'undersample':
        target_size = group_counts.min()
    elif strategy == 'balanced':
        target_size = int(group_counts.mean())
    else:
        raise ValueError(f"Unknown strategy: {strategy}")

    # Resample each group
    resampled_dfs = []
    for group in df_train['group'].unique():
        df_group = df_train[df_train['group'] == group]

        if len(df_group) < target_size:
            # Oversample with replacement
            df_group_resampled = resample(df_group,
                                         n_samples=target_size,
                                         replace=True,
                                         random_state=42)
        elif len(df_group) > target_size:
            # Undersample without replacement
            df_group_resampled = resample(df_group,
                                         n_samples=target_size,
                                         replace=False,
                                         random_state=42)
        else:
            df_group_resampled = df_group

        resampled_dfs.append(df_group_resampled)

    # Combine resampled groups
    df_resampled = pd.concat(resampled_dfs, ignore_index=True)

    # Shuffle
    df_resampled = df_resampled.sample(frac=1, random_state=42).reset_index(drop=True)

    print(f"\nAfter {strategy}:")
    print(df_resampled['group'].value_counts())

    # Extract X, y, groups
    X_resampled = df_resampled[feature_cols].values
    y_resampled = df_resampled['label'].values
    groups_resampled = df_resampled['group'].values

    return X_resampled, y_resampled, groups_resampled

# Apply oversampling
X_train_oversampled, y_train_oversampled, groups_train_oversampled = resample_for_fairness(
    X_train, y_train, groups_train, strategy='oversample'
)

Original training set:
group
light    9648
dark      852
Name: count, dtype: int64

After oversample:
group
dark     9648
light    9648
Name: count, dtype: int64


In [8]:
# Train model on oversampled data
model_oversampled = LogisticRegression(random_state=42, max_iter=1000)
model_oversampled.fit(X_train_oversampled, y_train_oversampled)

# Predictions on original test set
y_pred_oversampled = model_oversampled.predict(X_test)
y_prob_oversampled = model_oversampled.predict_proba(X_test)[:, 1]

# Evaluate
results_oversampled = evaluate_fairness(
    y_test, y_pred_oversampled, y_prob_oversampled, groups_test,
    model_name="PRE-PROCESSING: Oversampling"
)


PRE-PROCESSING: Oversampling - Fairness Evaluation

DARK SKIN (n=481)
  Prevalence: 0.100
  Sensitivity (TPR): 0.750  ⭐ CRITICAL for cancer detection
  Specificity (TNR): 0.972
  PPV (Precision): 0.750
  NPV: 0.972
  Accuracy: 0.950
  AUC: 0.955
  Selection Rate: 0.100
  Confusion: TP=36, FP=12, FN=12, TN=421

LIGHT SKIN (n=5519)
  Prevalence: 0.155
  Sensitivity (TPR): 0.943  ⭐ CRITICAL for cancer detection
  Specificity (TNR): 0.989
  PPV (Precision): 0.940
  NPV: 0.989
  Accuracy: 0.982
  AUC: 0.983
  Selection Rate: 0.155
  Confusion: TP=805, FP=51, FN=49, TN=4614

FAIRNESS GAPS:
  Sensitivity gap: 0.193 ⚠️ LARGE
  Specificity gap: 0.017 ✓ OK
  PPV gap: 0.190 ⚠️ LARGE
  AUC gap: 0.028 ✓ OK

⚠️ CLINICAL CONCERN: Large sensitivity gap detected!
   This means melanoma detection rates differ significantly by skin tone.
   Bias mitigation is strongly recommended.



### Technique 2: SMOTE (Synthetic Minority Over-sampling)

Instead of duplicating minority samples, SMOTE creates synthetic samples by interpolating between existing minority examples.

**How SMOTE works**:
1. For each minority sample, find k nearest neighbors
2. Create synthetic sample along line between sample and a random neighbor
3. New sample = sample + λ × (neighbor - sample), where λ ∈ [0,1]

In [9]:
# Apply SMOTE separately to each group
# We need to balance skin tone representation

# First, separate by group
mask_light = groups_train == 'light'
mask_dark = groups_train == 'dark'

X_train_light = X_train[mask_light]
y_train_light = y_train[mask_light]
X_train_dark = X_train[mask_dark]
y_train_dark = y_train[mask_dark]

print(f"Light skin training samples: {len(X_train_light)}")
print(f"Dark skin training samples: {len(X_train_dark)}")

# Apply SMOTE to minority class within each group
smote = SMOTE(random_state=42, k_neighbors=5)

# SMOTE for light skin group
if len(np.unique(y_train_light)) > 1:
    X_train_light_smote, y_train_light_smote = smote.fit_resample(X_train_light, y_train_light)
else:
    X_train_light_smote, y_train_light_smote = X_train_light, y_train_light

# SMOTE for dark skin group (with check for minimum samples)
if len(np.unique(y_train_dark)) > 1 and len(X_train_dark) > 6:
    X_train_dark_smote, y_train_dark_smote = smote.fit_resample(X_train_dark, y_train_dark)
else:
    X_train_dark_smote, y_train_dark_smote = X_train_dark, y_train_dark

print(f"\nAfter SMOTE:")
print(f"Light skin training samples: {len(X_train_light_smote)}")
print(f"Dark skin training samples: {len(X_train_dark_smote)}")

# Now balance group representation by oversampling dark skin
target_size = max(len(X_train_light_smote), len(X_train_dark_smote))

# Oversample dark skin to match light skin
if len(X_train_dark_smote) < target_size:
    indices = np.random.choice(len(X_train_dark_smote),
                              size=target_size - len(X_train_dark_smote),
                              replace=True)
    X_train_dark_smote = np.vstack([X_train_dark_smote, X_train_dark_smote[indices]])
    y_train_dark_smote = np.hstack([y_train_dark_smote, y_train_dark_smote[indices]])

# Combine groups
X_train_smote = np.vstack([X_train_light_smote, X_train_dark_smote])
y_train_smote = np.hstack([y_train_light_smote, y_train_dark_smote])
groups_train_smote = np.array(['light'] * len(X_train_light_smote) +
                              ['dark'] * len(X_train_dark_smote))

# Shuffle
shuffle_idx = np.random.permutation(len(X_train_smote))
X_train_smote = X_train_smote[shuffle_idx]
y_train_smote = y_train_smote[shuffle_idx]
groups_train_smote = groups_train_smote[shuffle_idx]

print(f"\nFinal SMOTE training set:")
print(f"Total samples: {len(X_train_smote)}")
print(f"Group distribution:")
print(pd.Series(groups_train_smote).value_counts())

Light skin training samples: 9648
Dark skin training samples: 852



After SMOTE:
Light skin training samples: 16388
Dark skin training samples: 1456

Final SMOTE training set:
Total samples: 32776
Group distribution:
light    16388
dark     16388
Name: count, dtype: int64


In [10]:
# Train model on SMOTE data
model_smote = LogisticRegression(random_state=42, max_iter=1000)
model_smote.fit(X_train_smote, y_train_smote)

# Predictions
y_pred_smote = model_smote.predict(X_test)
y_prob_smote = model_smote.predict_proba(X_test)[:, 1]

# Evaluate
results_smote = evaluate_fairness(
    y_test, y_pred_smote, y_prob_smote, groups_test,
    model_name="PRE-PROCESSING: SMOTE"
)


PRE-PROCESSING: SMOTE - Fairness Evaluation

DARK SKIN (n=481)
  Prevalence: 0.100
  Sensitivity (TPR): 0.854  ⭐ CRITICAL for cancer detection
  Specificity (TNR): 0.912
  PPV (Precision): 0.519
  NPV: 0.983
  Accuracy: 0.906
  AUC: 0.955
  Selection Rate: 0.164
  Confusion: TP=41, FP=38, FN=7, TN=395

LIGHT SKIN (n=5519)
  Prevalence: 0.155
  Sensitivity (TPR): 0.970  ⭐ CRITICAL for cancer detection
  Specificity (TNR): 0.960
  PPV (Precision): 0.817
  NPV: 0.994
  Accuracy: 0.962
  AUC: 0.983
  Selection Rate: 0.184
  Confusion: TP=828, FP=185, FN=26, TN=4480

FAIRNESS GAPS:
  Sensitivity gap: 0.115 ⚠️ LARGE
  Specificity gap: 0.048 ✓ OK
  PPV gap: 0.298 ⚠️ LARGE
  AUC gap: 0.028 ✓ OK

⚠️ CLINICAL CONCERN: Large sensitivity gap detected!
   This means melanoma detection rates differ significantly by skin tone.
   Bias mitigation is strongly recommended.



## Part 3: In-Processing Mitigation

**Strategy**: Incorporate fairness constraints directly into the model training objective.

We'll implement a fairness-constrained logistic regression with a custom loss function:

$$L_{\text{fair}} = L_{\text{standard}} + \lambda \cdot \text{fairness penalty}$$

The penalty targets **equal opportunity** (equal sensitivity), the criterion that matters most for cancer screening:

$$\text{fairness penalty} = \left(\widetilde{TPR}_{\text{light}} - \widetilde{TPR}_{\text{dark}}\right)^2, \qquad \widetilde{TPR}_g = \frac{1}{|P_g|}\sum_{i \in P_g} \hat{p}_i$$

where $P_g$ are the true melanomas in group $g$ and $\hat{p}_i$ the predicted **probability**. Two design choices matter:

- **Soft rates on probabilities, squared**: TPR computed from hard 0/1 predictions is a step function with zero gradient, so a gradient-based optimiser cannot follow it. The mean predicted probability is smooth, and squaring gives a smooth penalty that grows quickly for large gaps.
- **Group-specific intercepts**: skin tone enters only through a per-group intercept. A group-blind linear model can satisfy the penalty only by *levelling down* (pushing every probability toward 0); we show this below.

In [11]:
from sklearn.base import BaseEstimator, ClassifierMixin
from scipy.optimize import minimize
from itertools import combinations

class FairLogisticRegression(BaseEstimator, ClassifierMixin):
    """
    Logistic regression with an equal-opportunity (equal sensitivity) penalty.

    Loss = logistic loss + 1e-4·||w||² + λ · Σ_{group pairs a,b} (sTPR_a − sTPR_b)²

    sTPR_g is a *soft* true-positive rate: the mean predicted PROBABILITY over the true
    melanomas of group g. TPR from hard 0/1 predictions is a step function (zero gradient
    almost everywhere), so an optimiser cannot follow it; the soft rate is differentiable.

    group_intercepts=True gives each group its own intercept (skin tone is used ONLY there,
    never multiplied with lesion features). Without it, a group-blind linear model can only
    shrink the gap by "levelling down" (pushing everyone's probabilities toward 0).

    Parameters:
    - fairness_penalty: λ (strength of the sensitivity-equalisation penalty)
    """
    def __init__(self, fairness_penalty=1.0, group_intercepts=True, max_iter=1000):
        self.fairness_penalty = fairness_penalty
        self.group_intercepts = group_intercepts
        self.max_iter = max_iter

    def _sigmoid(self, z):
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def _design(self, X, groups):
        """Features + intercept column(s): one per group, or a single shared one."""
        if self.group_intercepts:
            return np.hstack([X, (np.asarray(groups)[:, None] == self.groups_[None, :]).astype(float)])
        return np.hstack([X, np.ones((X.shape[0], 1))])

    def _objective(self, w, X, y, groups):
        """Loss and its analytic gradient."""
        d = self.n_lesion_features_
        p = self._sigmoid(X @ w)
        pc = np.clip(p, 1e-15, 1 - 1e-15)
        loss = -np.mean(y * np.log(pc) + (1 - y) * np.log(1 - pc)) + 1e-4 * np.sum(w[:d]**2)
        grad = X.T @ (p - y) / len(y)
        grad[:d] += 2e-4 * w[:d]                   # intercepts are not regularised

        # Soft TPR per group and its gradient
        stpr, dstpr = {}, {}
        for g in self.groups_:
            pos = (groups == g) & (y == 1)
            if pos.sum() == 0:
                continue
            stpr[g] = p[pos].mean()
            dstpr[g] = (X[pos] * (p[pos] * (1 - p[pos]))[:, None]).mean(axis=0)
        for a, b in combinations(stpr, 2):
            diff = stpr[a] - stpr[b]
            loss += self.fairness_penalty * diff**2      # SQUARED sensitivity difference
            grad += self.fairness_penalty * 2 * diff * (dstpr[a] - dstpr[b])
        return loss, grad

    def fit(self, X, y, groups):
        """Fit fair logistic regression."""
        groups = np.asarray(groups)
        self.groups_ = np.unique(groups)
        self.n_lesion_features_ = X.shape[1]
        Xd = self._design(X, groups)
        result = minimize(self._objective, np.zeros(Xd.shape[1]), args=(Xd, y, groups),
                          jac=True, method='L-BFGS-B', options={'maxiter': self.max_iter})
        self.w_ = result.x
        self.coef_ = result.x[:X.shape[1]]
        return self

    def predict_proba(self, X, groups):
        """Predict class probabilities (groups needed for the group intercepts)."""
        p = self._sigmoid(self._design(X, groups) @ self.w_)
        return np.vstack([1 - p, p]).T

    def predict(self, X, groups):
        """Predict class labels at threshold 0.5."""
        return (self.predict_proba(X, groups)[:, 1] >= 0.5).astype(int)

print("✓ FairLogisticRegression class defined")

✓ FairLogisticRegression class defined


In [12]:
# Train fair model with a moderate fairness penalty
LAMBDA = 10.0
model_fair = FairLogisticRegression(fairness_penalty=LAMBDA)

print("Training fairness-constrained model...\n")
model_fair.fit(X_train, y_train, groups_train)

# Predictions
y_pred_fair = model_fair.predict(X_test, groups_test)
y_prob_fair = model_fair.predict_proba(X_test, groups_test)[:, 1]

# Evaluate
results_fair = evaluate_fairness(
    y_test, y_pred_fair, y_prob_fair, groups_test,
    model_name=f"IN-PROCESSING: Fairness-Constrained Model (λ={LAMBDA:g})"
)

# Why group intercepts? Same penalty, group-blind model
blind = FairLogisticRegression(fairness_penalty=LAMBDA, group_intercepts=False).fit(X_train, y_train, groups_train)
y_pred_blind = blind.predict(X_test, groups_test)
sens = lambda pred, g: pred[(groups_test == g) & (y_test == 1)].mean()
print(f"Group-blind model, same λ: sensitivity light {sens(y_pred_blind, 'light'):.3f}, dark {sens(y_pred_blind, 'dark'):.3f}")
print(f"Group-intercept model:     sensitivity light {sens(y_pred_fair, 'light'):.3f}, dark {sens(y_pred_fair, 'dark'):.3f}")
base_mean = (baseline_results['light']['sensitivity'] + baseline_results['dark']['sensitivity']) / 2
if (sens(y_pred_blind, 'light') + sens(y_pred_blind, 'dark')) / 2 < base_mean - 0.05:
    print("⚠️  The group-blind version 'levels down': it lowers sensitivity for everyone instead of raising it for dark skin.")

Training fairness-constrained model...




IN-PROCESSING: Fairness-Constrained Model (λ=10) - Fairness Evaluation

DARK SKIN (n=481)
  Prevalence: 0.100
  Sensitivity (TPR): 0.833  ⭐ CRITICAL for cancer detection
  Specificity (TNR): 0.921
  PPV (Precision): 0.541
  NPV: 0.980
  Accuracy: 0.913
  AUC: 0.956
  Selection Rate: 0.154
  Confusion: TP=40, FP=34, FN=8, TN=399

LIGHT SKIN (n=5519)
  Prevalence: 0.155
  Sensitivity (TPR): 0.915  ⭐ CRITICAL for cancer detection
  Specificity (TNR): 0.994
  PPV (Precision): 0.964
  NPV: 0.984
  Accuracy: 0.982
  AUC: 0.984
  Selection Rate: 0.147
  Confusion: TP=781, FP=29, FN=73, TN=4636

FAIRNESS GAPS:
  Sensitivity gap: 0.081 ⚠️ LARGE
  Specificity gap: 0.072 ⚠️ LARGE
  PPV gap: 0.424 ⚠️ LARGE
  AUC gap: 0.028 ✓ OK

⚠️ CLINICAL CONCERN: Large sensitivity gap detected!
   This means melanoma detection rates differ significantly by skin tone.
   Bias mitigation is strongly recommended.

Group-blind model, same λ: sensitivity light 0.660, dark 0.604
Group-intercept model:     sensitivit

## Part 4: Post-Processing Mitigation

**Strategy**: Keep the trained model fixed, but adjust decision thresholds per group to achieve fairness.

Instead of using a single threshold (e.g., 0.5) for all groups, we can use:
- Threshold for light skin: $t_L$
- Threshold for dark skin: $t_D$

Choose thresholds to equalize a fairness metric (e.g., TPR or FPR).

In [13]:
def find_optimal_thresholds(y_true, y_prob, groups, fairness_type='equalized_odds', min_specificity=0.0):
    """
    Find group-specific thresholds to satisfy fairness constraints.

    Approach:
    1. For each group, compute TPR and FPR at various thresholds
    2. Find threshold pair that minimizes fairness gap while maintaining utility,
       keeping specificity >= min_specificity in every group (rules out "flag everyone")

    Returns:
    - Dictionary of thresholds per group
    """
    unique_groups = np.unique(groups)

    # Compute TPR/FPR curves for each group
    group_curves = {}

    for group in unique_groups:
        mask = groups == group
        y_true_group = y_true[mask]
        y_prob_group = y_prob[mask]

        # Try many thresholds
        thresholds = np.linspace(0, 1, 101)
        tprs = []
        fprs = []
        accuracies = []

        for threshold in thresholds:
            y_pred = (y_prob_group >= threshold).astype(int)

            # Calculate metrics
            positives = y_true_group == 1
            negatives = y_true_group == 0

            if positives.sum() > 0:
                tpr = (y_pred[positives] == 1).mean()
            else:
                tpr = 0

            if negatives.sum() > 0:
                fpr = (y_pred[negatives] == 1).mean()
            else:
                fpr = 0

            accuracy = (y_pred == y_true_group).mean()

            tprs.append(tpr)
            fprs.append(fpr)
            accuracies.append(accuracy)

        group_curves[group] = {
            'thresholds': thresholds,
            'tprs': np.array(tprs),
            'fprs': np.array(fprs),
            'accuracies': np.array(accuracies)
        }

    # Now find optimal threshold pair
    # Objective: minimize fairness gap + maintain high overall TPR

    groups_list = list(unique_groups)
    best_thresholds = {}
    best_score = float('inf')

    # Grid search over threshold pairs
    for i1 in range(len(group_curves[groups_list[0]]['thresholds'])):
        for i2 in range(len(group_curves[groups_list[1]]['thresholds'])):
            t1 = group_curves[groups_list[0]]['thresholds'][i1]
            t2 = group_curves[groups_list[1]]['thresholds'][i2]

            tpr1 = group_curves[groups_list[0]]['tprs'][i1]
            tpr2 = group_curves[groups_list[1]]['tprs'][i2]
            fpr1 = group_curves[groups_list[0]]['fprs'][i1]
            fpr2 = group_curves[groups_list[1]]['fprs'][i2]

            if min(1 - fpr1, 1 - fpr2) < min_specificity:
                continue

            # Fairness gaps
            tpr_gap = abs(tpr1 - tpr2)
            fpr_gap = abs(fpr1 - fpr2)

            # Utility: mean TPR (we want high sensitivity for cancer detection)
            mean_tpr = (tpr1 + tpr2) / 2

            # Score: minimize gaps, maximize mean TPR
            if fairness_type == 'equalized_odds':
                score = (tpr_gap + fpr_gap) - mean_tpr  # Minimize gaps, maximize TPR
            elif fairness_type == 'equal_opportunity':
                score = tpr_gap - mean_tpr
            else:
                score = tpr_gap - mean_tpr

            if score < best_score:
                best_score = score
                best_thresholds = {
                    groups_list[0]: t1,
                    groups_list[1]: t2
                }

    return best_thresholds

# Tune thresholds on the VALIDATION set (never on the test set used for evaluation)
y_prob_val_baseline = baseline_model.predict_proba(X_val)[:, 1]
optimal_thresholds = find_optimal_thresholds(
    y_val, y_prob_val_baseline, groups_val,
    fairness_type='equal_opportunity', min_specificity=0.80
)

print("Optimal group-specific thresholds (tuned on the validation set, specificity ≥ 0.80 per group):")
for group, threshold in optimal_thresholds.items():
    print(f"  {group}: {threshold:.3f}")

if not optimal_thresholds:
    raise ValueError("No threshold pair keeps specificity ≥ 0.80 in both groups; relax min_specificity.")
for g, t in optimal_thresholds.items():
    m = groups_val == g
    pv = (y_prob_val_baseline[m] >= t)
    print(f"  validation {g}: sensitivity {pv[y_val[m] == 1].mean():.3f}, specificity {1 - pv[y_val[m] == 0].mean():.3f}")
print("The test set was not used to choose these thresholds; the next cell evaluates them on it.")

Optimal group-specific thresholds (tuned on the validation set, specificity ≥ 0.80 per group):
  dark: 0.040
  light: 0.660
  validation dark: sensitivity 0.906, specificity 0.821
  validation light: sensitivity 0.907, specificity 0.995
The test set was not used to choose these thresholds; the next cell evaluates them on it.


In [14]:
# Apply group-specific thresholds
y_pred_postprocess = np.zeros_like(y_test)

for group in np.unique(groups_test):
    mask = groups_test == group
    threshold = optimal_thresholds[group]
    y_pred_postprocess[mask] = (y_prob_baseline[mask] >= threshold).astype(int)

# Evaluate
results_postprocess = evaluate_fairness(
    y_test, y_pred_postprocess, y_prob_baseline, groups_test,
    model_name="POST-PROCESSING: Threshold Optimization"
)

# Did post-processing level DOWN instead of up?
for g in ['light', 'dark']:
    drop = baseline_results[g]['sensitivity'] - results_postprocess[g]['sensitivity']
    if drop > 0.05:
        print(f"⚠️ LEVELLING DOWN: {g}-skin sensitivity fell by {drop:.3f} vs baseline. The search (score = gap − mean TPR,")
        print("   specificity ≥ 0.80) equalised TPR partly by raising the other group's threshold, which harms those patients.")
val_gap = abs(np.mean(y_prob_val_baseline[(groups_val == 'light') & (y_val == 1)] >= optimal_thresholds['light'])
              - np.mean(y_prob_val_baseline[(groups_val == 'dark') & (y_val == 1)] >= optimal_thresholds['dark']))
test_gap = abs(results_postprocess['light']['sensitivity'] - results_postprocess['dark']['sensitivity'])
n_dark_pos_val = int(((groups_val == 'dark') & (y_val == 1)).sum())
print(f"Sensitivity gap: validation {val_gap:.3f} → test {test_gap:.3f} "
      f"(validation had only {n_dark_pos_val} dark-skin melanomas to tune on).")
if test_gap > val_gap + 0.05:
    print("   The thresholds did not transfer: with so few dark-skin cases, tuned thresholds overfit the validation set.")


POST-PROCESSING: Threshold Optimization - Fairness Evaluation

DARK SKIN (n=481)
  Prevalence: 0.100
  Sensitivity (TPR): 0.896  ⭐ CRITICAL for cancer detection
  Specificity (TNR): 0.831
  PPV (Precision): 0.371
  NPV: 0.986
  Accuracy: 0.838
  AUC: 0.956
  Selection Rate: 0.241
  Confusion: TP=43, FP=73, FN=5, TN=360

LIGHT SKIN (n=5519)
  Prevalence: 0.155
  Sensitivity (TPR): 0.900  ⭐ CRITICAL for cancer detection
  Specificity (TNR): 0.994
  PPV (Precision): 0.966
  NPV: 0.982
  Accuracy: 0.980
  AUC: 0.984
  Selection Rate: 0.144
  Confusion: TP=769, FP=27, FN=85, TN=4638

FAIRNESS GAPS:
  Sensitivity gap: 0.005 ✓ OK
  Specificity gap: 0.163 ⚠️ LARGE
  PPV gap: 0.595 ⚠️ LARGE
  AUC gap: 0.028 ✓ OK

Sensitivity gap: validation 0.001 → test 0.005 (validation had only 32 dark-skin melanomas to tune on).


## Part 5: Comparison of Mitigation Strategies

Let's compare all approaches side-by-side.

In [15]:
# Compile results
all_results = {
    'Baseline (No Mitigation)': baseline_results,
    'Pre: Oversampling': results_oversampled,
    'Pre: SMOTE': results_smote,
    'In: Fair Model': results_fair,
    'Post: Threshold Opt': results_postprocess
}

# Create comparison DataFrame
comparison_data = []

for method_name, results in all_results.items():
    for group in ['light', 'dark']:
        if group in results:
            r = results[group]
            comparison_data.append({
                'Method': method_name,
                'Group': group,
                'Sensitivity': r['sensitivity'],
                'Specificity': r['specificity'],
                'PPV': r['ppv'],
                'Accuracy': r['accuracy'],
                'AUC': r['auc']
            })

df_comparison = pd.DataFrame(comparison_data)

print("\n" + "="*80)
print("COMPREHENSIVE COMPARISON OF BIAS MITIGATION STRATEGIES")
print("="*80 + "\n")

# Display table
print(df_comparison.to_string(index=False))
print()


COMPREHENSIVE COMPARISON OF BIAS MITIGATION STRATEGIES

                  Method Group  Sensitivity  Specificity      PPV  Accuracy      AUC
Baseline (No Mitigation) light     0.942623     0.991211 0.951537  0.983693 0.984077
Baseline (No Mitigation)  dark     0.708333     0.974596 0.755556  0.948025 0.955735
       Pre: Oversampling light     0.942623     0.989068 0.940421  0.981881 0.983087
       Pre: Oversampling  dark     0.750000     0.972286 0.750000  0.950104 0.955446
              Pre: SMOTE light     0.969555     0.960343 0.817374  0.961768 0.982931
              Pre: SMOTE  dark     0.854167     0.912240 0.518987  0.906445 0.955254
          In: Fair Model light     0.914520     0.993783 0.964198  0.981518 0.983681
          In: Fair Model  dark     0.833333     0.921478 0.540541  0.912682 0.956168
     Post: Threshold Opt light     0.900468     0.994212 0.966080  0.979706 0.984077
     Post: Threshold Opt  dark     0.895833     0.831409 0.370690  0.837838 0.955735



In [16]:
# Visualize comparison
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = ['Sensitivity', 'Specificity', 'PPV', 'AUC']
metric_titles = ['Sensitivity (TPR)', 'Specificity (TNR)', 'PPV (Precision)', 'AUC']

for idx, (metric, title) in enumerate(zip(metrics, metric_titles)):
    ax = axes[idx // 2, idx % 2]

    # Prepare data for grouped bar chart
    methods = df_comparison['Method'].unique()
    light_values = []
    dark_values = []

    for method in methods:
        light_val = df_comparison[(df_comparison['Method'] == method) &
                                 (df_comparison['Group'] == 'light')][metric].values[0]
        dark_val = df_comparison[(df_comparison['Method'] == method) &
                                (df_comparison['Group'] == 'dark')][metric].values[0]
        light_values.append(light_val)
        dark_values.append(dark_val)

    x = np.arange(len(methods))
    width = 0.35

    bars1 = ax.bar(x - width/2, light_values, width, label='Light Skin',
                   color='lightblue', edgecolor='black')
    bars2 = ax.bar(x + width/2, dark_values, width, label='Dark Skin',
                   color='darkblue', edgecolor='black')

    # Add gap annotations
    for i, (l, d) in enumerate(zip(light_values, dark_values)):
        gap = abs(l - d)
        y_pos = max(l, d) + 0.02
        color = 'green' if gap < 0.05 else 'red'
        ax.text(i, y_pos, f'Δ={gap:.3f}', ha='center', fontsize=9,
               color=color, fontweight='bold')

    ax.set_ylabel(title, fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(methods, rotation=45, ha='right', fontsize=10)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 1.1)

    # Add fairness threshold line
    ax.axhline(y=0.8, color='gray', linestyle='--', alpha=0.5, label='Clinical Target')

plt.tight_layout()
plt.savefig('mitigation_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Key observations, computed from df_comparison
def _row(method, group):
    return df_comparison[(df_comparison['Method'] == method) & (df_comparison['Group'] == group)].iloc[0]

base = 'Baseline (No Mitigation)'
print("\nKey Observations (computed from the table above):")
for m in methods:
    l, d = _row(m, 'light'), _row(m, 'dark')
    note = ""
    if m != base:
        delta = d['Sensitivity'] - _row(base, 'dark')['Sensitivity']
        note = f"; dark-skin sensitivity {'+' if delta >= 0 else ''}{delta:.3f} vs baseline"
    if min(l['Sensitivity'], d['Sensitivity']) < 0.05 or min(l['Specificity'], d['Specificity']) < 0.05:
        note += "  ⚠️ DEGENERATE (flags everyone or no one)"
    print(f"- {m:26s}: sensitivity gap {abs(l['Sensitivity'] - d['Sensitivity']):.3f}{note}")
d_over = _row('Pre: Oversampling', 'dark')['Sensitivity'] - _row(base, 'dark')['Sensitivity']
if d_over < 0.02:
    print("\nOversampling barely moved dark-skin sensitivity: duplicating rows adds no new information.")
else:
    print(f"\nOversampling raised dark-skin sensitivity by {d_over:.3f}: giving dark-skin rows more weight shifts the")
    print("group-blind model's decision boundary toward their (weaker) melanoma cues.")
print("In this generator the dark-skin gap comes from weaker, noisier visual cues: the same features score")
print("dark-skin melanomas lower. Shifts like that are what group-aware intercepts or thresholds can correct.")


Key Observations (computed from the table above):
- Baseline (No Mitigation)  : sensitivity gap 0.234
- Pre: Oversampling         : sensitivity gap 0.193; dark-skin sensitivity +0.042 vs baseline
- Pre: SMOTE                : sensitivity gap 0.115; dark-skin sensitivity +0.146 vs baseline
- In: Fair Model            : sensitivity gap 0.081; dark-skin sensitivity +0.125 vs baseline
- Post: Threshold Opt       : sensitivity gap 0.005; dark-skin sensitivity +0.188 vs baseline

Oversampling raised dark-skin sensitivity by 0.042: giving dark-skin rows more weight shifts the
group-blind model's decision boundary toward their (weaker) melanoma cues.
In this generator the dark-skin gap comes from weaker, noisier visual cues: the same features score
dark-skin melanomas lower. Shifts like that are what group-aware intercepts or thresholds can correct.


## Part 6: Fairness-Accuracy Trade-off Analysis

A critical question: **What is the cost of fairness?**

Let's explore the trade-off between fairness and overall accuracy.

In [17]:
# Train models with varying fairness penalties
fairness_penalties = [0.0, 0.5, 1.0, 2.0, 5.0, 10.0, 20.0, 50.0]
tradeoff_results = []

print("Exploring fairness-accuracy trade-off...")
print("(Training multiple models with different fairness penalties)\n")

for penalty in fairness_penalties:
    print(f"Training model with fairness_penalty={penalty}...")

    model = FairLogisticRegression(fairness_penalty=penalty)
    model.fit(X_train, y_train, groups_train)

    y_pred = model.predict(X_test, groups_test)
    y_prob = model.predict_proba(X_test, groups_test)[:, 1]

    # Calculate metrics per group
    for group in ['light', 'dark']:
        mask = groups_test == group
        y_true_group = y_test[mask]
        y_pred_group = y_pred[mask]

        tn, fp, fn, tp = confusion_matrix(y_true_group, y_pred_group).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        accuracy = (tp + tn) / (tp + tn + fp + fn)

        tradeoff_results.append({
            'fairness_penalty': penalty,
            'group': group,
            'sensitivity': sensitivity,
            'specificity': specificity,
            'accuracy': accuracy,
            'soft_tpr': y_prob[mask & (y_test == 1)].mean()   # what the penalty actually equalises
        })

df_tradeoff = pd.DataFrame(tradeoff_results)

# Calculate gaps
tradeoff_gaps = []
for penalty in fairness_penalties:
    light = df_tradeoff[(df_tradeoff['fairness_penalty'] == penalty) &
                       (df_tradeoff['group'] == 'light')].iloc[0]
    dark = df_tradeoff[(df_tradeoff['fairness_penalty'] == penalty) &
                      (df_tradeoff['group'] == 'dark')].iloc[0]

    sens_gap = abs(light['sensitivity'] - dark['sensitivity'])
    mean_acc = (light['accuracy'] + dark['accuracy']) / 2
    mean_sens = (light['sensitivity'] + dark['sensitivity']) / 2

    tradeoff_gaps.append({
        'fairness_penalty': penalty,
        'sensitivity_gap': sens_gap,
        'mean_accuracy': mean_acc,
        'mean_sensitivity': mean_sens,
        'soft_tpr_gap': abs(light['soft_tpr'] - dark['soft_tpr']),
        'accuracy': (light['accuracy'] * (groups_test == 'light').sum() +
                     dark['accuracy'] * (groups_test == 'dark').sum()) / len(groups_test)
    })

df_gaps = pd.DataFrame(tradeoff_gaps)

print("\n✓ Trade-off analysis complete")

Exploring fairness-accuracy trade-off...
(Training multiple models with different fairness penalties)

Training model with fairness_penalty=0.0...
Training model with fairness_penalty=0.5...
Training model with fairness_penalty=1.0...


Training model with fairness_penalty=2.0...
Training model with fairness_penalty=5.0...
Training model with fairness_penalty=10.0...


Training model with fairness_penalty=20.0...
Training model with fairness_penalty=50.0...

✓ Trade-off analysis complete


In [18]:
# Visualize trade-off
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Sensitivity gap vs fairness penalty
ax1 = axes[0]
ax1.plot(df_gaps['fairness_penalty'], df_gaps['sensitivity_gap'],
         marker='o', linewidth=2, markersize=8, color='red', label='Sensitivity Gap')
ax1.axhline(y=0.05, color='green', linestyle='--', label='Fairness Threshold (5%)')
ax1.set_xlabel('Fairness Penalty (λ)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Sensitivity Gap (|TPR_light - TPR_dark|)', fontsize=12, fontweight='bold')
ax1.set_title('Fairness Improvement vs Penalty Strength', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Mean sensitivity vs sensitivity gap (Pareto frontier)
ax2 = axes[1]
scatter = ax2.scatter(df_gaps['sensitivity_gap'], df_gaps['mean_sensitivity'],
                     c=df_gaps['fairness_penalty'], s=200, cmap='viridis',
                     edgecolors='black', linewidth=1.5)
cbar = plt.colorbar(scatter, ax=ax2)
cbar.set_label('Fairness Penalty (λ)', fontsize=11)

# Annotate points
for idx, row in df_gaps.iterrows():
    ax2.annotate(f"λ={row['fairness_penalty']:.0f}",
                (row['sensitivity_gap'], row['mean_sensitivity']),
                fontsize=9, ha='center', va='bottom')

ax2.axvline(x=0.05, color='green', linestyle='--', alpha=0.5, label='Fairness Threshold')
ax2.set_xlabel('Sensitivity Gap', fontsize=12, fontweight='bold')
ax2.set_ylabel('Mean Sensitivity (Clinical Utility)', fontsize=12, fontweight='bold')
ax2.set_title('Fairness-Utility Trade-off (Pareto Frontier)', fontsize=14, fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('fairness_accuracy_tradeoff.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nKey Insights (computed from the sweep):")
print(df_gaps.round(3).to_string(index=False))
first, last = df_gaps.iloc[0], df_gaps.iloc[-1]
print(f"\n1. Sensitivity gap: {first['sensitivity_gap']:.3f} at λ={first['fairness_penalty']:g} → "
      f"{last['sensitivity_gap']:.3f} at λ={last['fairness_penalty']:g}")
print(f"2. Mean sensitivity: {first['mean_sensitivity']:.3f} → {last['mean_sensitivity']:.3f}")
print(f"3. Overall accuracy: {first['accuracy']:.3f} → {last['accuracy']:.3f} "
      f"({(last['accuracy'] - first['accuracy']) * 100:+.1f} percentage points)")
collapsed = df_gaps[(df_gaps['fairness_penalty'] > 0) & (df_gaps['mean_sensitivity'] < first['mean_sensitivity'] - 0.10)]
if len(collapsed):
    print(f"⚠️ For λ in {sorted(collapsed['fairness_penalty'].tolist())} mean sensitivity fell by >10 pp: the gap closes partly")
    print("   by levelling down. Prefer the smallest λ that brings the gap under the target.")
print(f"   Soft-TPR gap (the penalised quantity): {first['soft_tpr_gap']:.3f} → {last['soft_tpr_gap']:.3f}")
soft_cut = 1 - last['soft_tpr_gap'] / max(first['soft_tpr_gap'], 1e-9)
hard_cut = 1 - last['sensitivity_gap'] / max(first['sensitivity_gap'], 1e-9)
if soft_cut > 0.5 and hard_cut < 0.5:
    print(f"⚠️ The penalty removed {soft_cut:.0%} of the SOFT gap but only {hard_cut:.0%} of the hard-threshold sensitivity gap.")
    print("   The surrogate is a mean probability: the model raised mid-range dark-skin probabilities, but dark-skin")
    print("   melanomas with very low scores (weak cues, or mislabelled) are not pushed over 0.5. Thresholds and")
    print("   better data are still needed.")
small = df_gaps[df_gaps['sensitivity_gap'] < 0.05]
if len(small):
    print(f"   Smallest λ with sensitivity gap < 0.05: λ={small['fairness_penalty'].min():g}")
print("   Note: the test set has few dark-skin melanomas, so each gap value moves in steps of "
      f"{1 / max(1, ((groups_test == 'dark') & (y_test == 1)).sum()):.3f}.")
print("   (λ is shown on the test set here to visualise the trade-off; choose λ on validation data in practice.)")
usable = df_gaps[df_gaps['mean_sensitivity'] >= 0.80]
if len(usable):
    best = usable.loc[usable['sensitivity_gap'].idxmin()]
    print(f"4. Smallest gap with mean sensitivity ≥ 0.80: λ={best['fairness_penalty']:g} (gap {best['sensitivity_gap']:.3f})")
else:
    print("4. No λ keeps mean sensitivity ≥ 0.80 at the default 0.5 threshold in this sweep.")


Key Insights (computed from the sweep):
 fairness_penalty  sensitivity_gap  mean_accuracy  mean_sensitivity  soft_tpr_gap  accuracy
              0.0            0.208          0.965             0.833         0.156     0.981
              0.5            0.094          0.953             0.880         0.048     0.978
              1.0            0.089          0.952             0.878         0.030     0.978
              2.0            0.085          0.948             0.876         0.018     0.976
              5.0            0.081          0.947             0.874         0.009     0.976
             10.0            0.081          0.947             0.874         0.006     0.976
             20.0            0.080          0.947             0.873         0.004     0.976
             50.0            0.080          0.947             0.873         0.003     0.976

1. Sensitivity gap: 0.208 at λ=0 → 0.080 at λ=50
2. Mean sensitivity: 0.833 → 0.873
3. Overall accuracy: 0.981 → 0.976 (-0.5 perce

## Part 7: Clinical Decision-Making Guidance

### Which Mitigation Strategy to Use?

The choice depends on your constraints and priorities:

| **Strategy** | **Pros** | **Cons** | **When to Use** |
|--------------|----------|----------|------------------|
| **Pre-processing** | Easy to implement; works with any model | May not fully eliminate bias; can lose data (undersampling) | When you have control over training data; quick baseline solution |
| **In-processing** | Directly optimizes fairness; principled approach | Requires custom training; computationally expensive | When training from scratch; need guaranteed fairness |
| **Post-processing** | Works with any trained model; no retraining needed | Group-specific thresholds can be controversial | When model is already deployed; regulatory constraints |

### Clinical Context Considerations

**For melanoma detection (cancer screening)**:
- **Priority**: Minimize false negatives (missed cancers)
- **Fairness metric**: Equal opportunity (equal TPR/sensitivity)
- **Recommendation**: Post-processing with lower thresholds for underserved groups

**For resource allocation (e.g., high-risk patient identification)**:
- **Priority**: Balance TPR and FPR (limited resources)
- **Fairness metric**: Equalized odds (equal TPR and FPR)
- **Recommendation**: In-processing with fairness constraints

**For diagnostic support (e.g., imaging interpretation)**:
- **Priority**: Maintain high PPV (avoid alarm fatigue)
- **Fairness metric**: Predictive parity (equal PPV)
- **Recommendation**: Pre-processing + post-processing combination

In [19]:
# Summarize recommendations
print("="*80)
print("CLINICAL DEPLOYMENT RECOMMENDATIONS")
print("="*80 + "\n")

print("Based on our melanoma detection scenario (computed from the results above):\n")

MIN_SENS, MIN_SPEC = 0.80, 0.80
print(f"{'Method':28s} {'min sens':>9s} {'min spec':>9s} {'sens gap':>9s}")
summary = {}
for name, res in all_results.items():
    sens = [res[g]['sensitivity'] for g in res]
    spec = [res[g]['specificity'] for g in res]
    summary[name] = (min(sens), min(spec), max(sens) - min(sens))
    print(f"{name:28s} {min(sens):9.3f} {min(spec):9.3f} {max(sens) - min(sens):9.3f}")

acceptable = {m: v for m, v in summary.items() if v[0] >= MIN_SENS and v[1] >= MIN_SPEC}
if acceptable:
    best = min(acceptable, key=lambda m: acceptable[m][2])
    print(f"\n✓ Best option in THIS run: {best} — smallest sensitivity gap among methods with "
          f"≥{MIN_SENS:.0%} sensitivity and ≥{MIN_SPEC:.0%} specificity in every group.")
else:
    print(f"\n⚠️  NO method here reaches ≥{MIN_SENS:.0%} sensitivity AND ≥{MIN_SPEC:.0%} specificity in every group.")
    for m, (s, sp, gap) in summary.items():
        if sp < 0.05:
            print(f"   - {m}: gap {gap:.3f}, but specificity {sp:.2f} (flags (almost) everyone)")
        elif s < 0.05:
            print(f"   - {m}: gap {gap:.3f}, but sensitivity {s:.2f} (misses (almost) every melanoma)")
        else:
            print(f"   - {m}: gap {gap:.3f}, min sensitivity {s:.2f}, min specificity {sp:.2f}")
    print("   Do not deploy any of these as-is. Next steps: combine the fairness penalty with validation-tuned")
    print("   thresholds, and get better dark-skin data (more images, less label noise).")

print("\nIn principle, post-processing (per-group thresholds) suits screening because it needs no retraining and")
print("the thresholds are easy to explain, but only if each threshold keeps specificity clinically acceptable.\n")

print("📋 DEPLOYMENT CHECKLIST:")
print("  1. Validate on external test set with diverse skin tones")
print("  2. Monitor sensitivity gap in production (target <5%)")
print("  3. Ensure minimum sensitivity >80% for all groups")
print("  4. Collect feedback from dermatologists on false negatives")
print("  5. Re-evaluate fairness quarterly as data distribution shifts")
print("  6. Document mitigation strategy in model card")
print("  7. Obtain ethics review approval before clinical deployment\n")

CLINICAL DEPLOYMENT RECOMMENDATIONS

Based on our melanoma detection scenario (computed from the results above):

Method                        min sens  min spec  sens gap
Baseline (No Mitigation)         0.708     0.975     0.234
Pre: Oversampling                0.750     0.972     0.193
Pre: SMOTE                       0.854     0.912     0.115
In: Fair Model                   0.833     0.921     0.081
Post: Threshold Opt              0.896     0.831     0.005

✓ Best option in THIS run: Post: Threshold Opt — smallest sensitivity gap among methods with ≥80% sensitivity and ≥80% specificity in every group.

In principle, post-processing (per-group thresholds) suits screening because it needs no retraining and
the thresholds are easy to explain, but only if each threshold keeps specificity clinically acceptable.

📋 DEPLOYMENT CHECKLIST:
  1. Validate on external test set with diverse skin tones
  2. Monitor sensitivity gap in production (target <5%)
  3. Ensure minimum sensitivity >80

## Summary and Key Takeaways

### What We Learned

1. **Three Categories of Bias Mitigation**:
   - **Pre-processing**: Resampling, SMOTE to balance data
   - **In-processing**: Fairness-constrained training objectives
   - **Post-processing**: Group-specific threshold optimization

2. **Fairness-Accuracy Trade-off**:
   - Improving fairness often requires accepting lower overall accuracy
   - The trade-off is not always severe: in the λ sweep (Part 6) mean sensitivity rose while overall accuracy fell by less than 1 percentage point
   - Equal metrics are not automatically good: a model that flags everyone (or no one) has zero gaps (the notebook's DEGENERATE checks would flag this; none fired in this run)
   - Choice of fairness metric matters for clinical context

3. **No Universal Solution**:
   - Best mitigation strategy depends on:
     - Clinical application (screening vs diagnosis vs resource allocation)
     - Available resources (can we retrain? collect more data?)
     - Stakeholder priorities (which metrics matter most?)
     - Regulatory constraints (are group-specific thresholds allowed?)

4. **Practical Implementation**:
   - Start with baseline evaluation to quantify bias
   - Try multiple mitigation strategies
   - Evaluate on held-out test set with diverse representation
   - Involve clinicians and affected communities in decision-making
   - Monitor fairness continuously in production

### Critical Considerations

⚠️ **Bias mitigation is necessary but not sufficient**:
- Technical fairness ≠ real-world equity
- Must address root causes: data collection bias, healthcare access disparities
- Algorithmic fairness is one piece of a larger puzzle

⚠️ **Beware of fairness washing**:
- Don't use mitigation as excuse to deploy biased models
- Transparency: document limitations and residual disparities
- Continuous monitoring: fairness degrades as data distributions shift

---

## Exercises

1. **Undersample or balance instead**: In Part 2, call `resample_for_fairness(..., strategy='undersample')` and `strategy='balanced'`, retrain the `LogisticRegression` and evaluate with `evaluate_fairness`. Does discarding light-skin rows move dark-skin sensitivity more or less than oversampling did? What happens to light-skin specificity and PPV?

2. **Which fairness criterion, which floor?**: In Part 4, rerun `find_optimal_thresholds` with `fairness_type='equalized_odds'`, then with `min_specificity` set to 0.70 and 0.90. Track the dark-skin threshold and the test sensitivity, specificity and PPV for dark skin. Hint: a higher specificity floor forces a higher dark-skin threshold and gives back some of the sensitivity gain.

3. **Turn off one bias mechanism at a time**: In `create_melanoma_dataset`, set `DARK_CONTRAST = 1.0` and `DARK_EXTRA_SD = 0.0` but keep the 92/8 split, then rerun Parts 1–5. Then do the reverse: keep the knobs and make the split 50/50. Which mechanism produces most of the baseline sensitivity gap in this generator? Compare with the computed note under the Part 5 plot.

4. **Choose λ without peeking (deeper)**: Part 6 sweeps λ on the test set only to draw the trade-off. Rewrite it to pick λ on `X_val` / `groups_val` (for example, the smallest λ whose validation sensitivity gap is below 0.05, as Notebook 9.4 does) and evaluate that single model once on the test set. Then bootstrap the test rows (about 1,000 resamples) to get a 95% CI for the dark-skin sensitivity. With only 48 dark-skin melanomas in the test set, can you tell the in-processing and post-processing gaps apart?

5. **Clinical reflection: who pays for the lower threshold?**: Post-processing closed the sensitivity gap by flagging about a quarter of dark-skin patients, with a dark-skin PPV below 0.4 (Part 7 table). In Chapter 3, Priya skipped a dermatologist visit after a smartphone app told her "low risk". For an app like hers, would you accept more false alarms for dark-skin users to catch more melanomas? Who bears each kind of error, and should users be told that thresholds differ by skin tone?

---

### Next Steps

In the next notebook, we'll apply these mitigation strategies to **Priya's Journey**—our melanoma detection case study—and see how fairness interventions can reduce harm in a realistic clinical scenario.

---

**Further Reading**:
- Agarwal et al. (2018). "A Reductions Approach to Fair Classification"
- Hardt et al. (2016). "Equality of Opportunity in Supervised Learning"
- Kamiran & Calders (2012). "Data preprocessing techniques for classification without discrimination"
- Pleiss et al. (2017). "On Fairness and Calibration"

**Code Repositories**:
- Fairlearn: https://fairlearn.org/
- AI Fairness 360: https://aif360.mybluemix.net/
- Google's What-If Tool: https://pair-code.github.io/what-if-tool/